In [0]:
%run ../logging_utils

In [0]:
# %run ../../application_utils/metrics_custom

In [0]:
import json
import ast
import logging
from datetime import datetime
import pyspark.sql.functions as F
import pyspark.sql.types as T
from dateutil.relativedelta import relativedelta

In [0]:
import ast
import json
from datetime import datetime

RunId = dbutils.widgets.get("RunId")
cod_workflowName = dbutils.widgets.get("cod_workflowName")
cod_pipelineID = dbutils.widgets.get("cod_pipelineID")
cod_layerName = dbutils.widgets.get("cod_layerName")
cod_productName = dbutils.widgets.get("cod_productName")
snapshot_date = dbutils.widgets.get("snapshot_date")
input_params = dbutils.widgets.get("input_params")
env = dbutils.widgets.get("env")

general_utils = LoggingUtils(spark)
logger = general_utils.set_logging()

config = general_utils.load_config(general_utils.get_variable(cod_productName)['config_setup_path'])
snapshot_date = general_utils.date_to_start_month(snapshot_date)
catalog_name = config['environment'][env]['catalog']
schema_name = config['environment'][env]['schema']

try:
    input_params_json = ast.literal_eval(input_params)
except (ValueError, SyntaxError) as e:
    input_params_json = json.loads(input_params)

try:
    months_to_subtract = int(input_params_json.get('today_minus_n_months', 0))
    snapshot_date = general_utils.replace_month_date(snapshot_date, months_to_subtract)
except Exception:
    snapshot_date = general_utils.replace_month_date(snapshot_date, 0)

logger.info(f"Snapshot_date materialized: {snapshot_date}")

try:
    logger.info(f"MASTER Log generation process INITIALIZING")
    # logger.info(f"Get parameters")
    # params = general_utils.get_pipeline_parameters()

    logger.info(f"Get table reference")
    log_catalog, log_database, log_table = general_utils.get_log_processor_reference(
        catalog_name, 
        "monitoring", 
        config['logging']['processor']['table']
    )
    
    full_table_path = f"{log_catalog}.{log_database}.{log_table}"

except Exception as e:
    logger.error(f"Log generation process ERROR as {e}")
    raise

try:
    metric_name = input_params_json.get('name', 'UNKNOWN')
    metric_type = input_params_json.get('type', '')
    
    logger.info(f"MASTER Log generation process START for {metric_name}")
    
    if metric_type == cod_layerName:
        logger.info(f"START log writing for metrica: {metric_name}")
        
        now_date = datetime.now().strftime("%Y-%m-%d")
        now_timestamp = datetime.now().strftime("%Y-%m-%d %H:%M:%S")
        
        def sql_val(val):
            return f"'{val}'" if val is not None and str(val).strip() != "" else "NULL"
        
        sql_query = f"""
            INSERT INTO {full_table_path} (
                cod_workflowName, cod_layerName, dat_processDate, dat_processStartDate, 
                dat_processEndDate, cod_pipelineID, cod_runID, cod_productname, 
                cod_flowName, cod_flowNameSub, dat_snapshotDate, cod_processStatus, 
                cod_processErrorCode, des_processErrorDes
            ) VALUES (
                {sql_val(cod_workflowName)}, 
                {sql_val(metric_type)}, 
                CAST({sql_val(now_date)} AS DATE), 
                CAST({sql_val(now_timestamp)} AS TIMESTAMP), 
                NULL, 
                {sql_val(cod_pipelineID)}, 
                {sql_val(RunId)}, 
                {sql_val(cod_productName)}, 
                {sql_val(metric_name)}, 
                NULL, 
                CAST({sql_val(snapshot_date)} AS DATE), 
                'Started', 
                NULL, 
                NULL
            )
        """
        
        logger.info(f"Writing dataframe (via SQL) on {full_table_path} table")
        spark.sql(sql_query)
        logger.info(f"Log generation process END for {metric_name}")
        
    else:
        logger.info(f"Skipping log insertion: MASTER type '{metric_type}' does not match layer '{cod_layerName}'")

except Exception as e:
    logger.error(f"Log generation process ERROR as {e}")
    raise

In [0]:
# RunId = dbutils.widgets.get("RunId")
# cod_workflowName = dbutils.widgets.get("cod_workflowName")
# cod_pipelineID = dbutils.widgets.get("cod_pipelineID")
# cod_layerName = dbutils.widgets.get("cod_layerName")
# cod_productName = dbutils.widgets.get("cod_productName")
# snapshot_date=dbutils.widgets.get("snapshot_date")
# input_params=dbutils.widgets.get("input_params")
# env = dbutils.widgets.get("env")

# general_utils = LoggingUtils(spark)
# # metric_materializer = MetricMaterializer(spark)
# logger = general_utils.set_logging()

# logger.info(f"MASTER Log generation process INITIALIZING")
# config = general_utils.load_config(general_utils.get_variable()['config_setup_path'])

# snapshot_date = general_utils.date_to_start_month(snapshot_date)
# catalog_name = config['environment'][env]['catalog']
# schema_name = config['environment'][env]['schema']

# try:
#     input_params_json = ast.literal_eval(input_params)
# except (ValueError, SyntaxError) as e:
#     input_params_json = json.loads(input_params)

# try:
#     snapshot_date = general_utils.replace_month_date(snapshot_date, int(input_params_json['today_minus_n_months']))
# except:
#     snapshot_date = general_utils.replace_month_date(snapshot_date, 0)
# logger.info(f"Snapshot_date materialized: {snapshot_date}")


# try:
#     logger.info(f"Get parameters")
#     params = general_utils.get_pipeline_parameters()

#     logger.info(f"Get table reference")
#     log_catalog, log_database, log_table = general_utils.get_log_processor_reference(catalog_name, 
#                                 "monitoring", 
#                                 config['logging']['processor']['table'])

#     logger.info(f"Formatting table reference")
#     try:
#         input_params_json = ast.literal_eval(input_params)
#     except (ValueError, SyntaxError) as e:
#         input_params_json = json.loads(input_params)

# except Exception as e:
#     logger.error(f"Log generation process ERROR as {e}")
#     raise

# try:
#     logger.info(f"MASTER Log generation process START for {input_params_json['name']}")
#     if input_params_json['type'] == params['cod_layerName']:
#         logger.info(f"START log writing for metrica: {input_params_json['name']}")
#         lg_processmonitor_data = general_utils.get_processor_data(
#             # iid_processID= max_iid,
#             cod_workflowName=cod_workflowName,
#             cod_layerName=input_params_json['type'],
#             dat_processDate=datetime.now().strftime("%Y-%m-%d"),
#             dat_processStartDate=datetime.now().strftime("%Y-%m-%d %H:%M:%S"),
#             dat_processEndDate=None,
#             cod_pipelineID=cod_pipelineID,
#             cod_runID=RunId,
#             cod_productname=cod_productName,
#             cod_flowName=input_params_json['name'],
#             cod_flowNameSub=None,
#             dat_snapshotDate=snapshot_date,
#             cod_processStatus="Started",
#             cod_processErrorCode=None,
#             des_processErrorDes=None)

#         logger.info(f"Creating log dataframe")
#         lg_processmonitor = general_utils.get_lg_processmonitor_metadata(catalog_name, log_database, log_table, lg_processmonitor_data)

#         lg_processmonitor = lg_processmonitor.withColumn("dat_processDate",      F.col("dat_processDate")     .cast(T.DateType()))
#         lg_processmonitor = lg_processmonitor.withColumn("dat_processStartDate", F.col("dat_processStartDate").cast(T.TimestampType()))
#         lg_processmonitor = lg_processmonitor.withColumn("dat_processEndDate",   F.col("dat_processEndDate")  .cast(T.TimestampType()))
#         lg_processmonitor = lg_processmonitor.withColumn("dat_snapshotDate",     F.col("dat_snapshotDate")    .cast(T.DateType()))

#         logger.info(f"Writing dataframe on {log_table} table")
#         general_utils.write_table(catalog_name, log_database, log_table, lg_processmonitor, "append")

#         logger.info(f"Log generation process END for {input_params_json['name']}")
# except Exception as e:
#     logger.error(f"Log generation process ERROR as {e}")
#     raise

In [0]:
dbutils.notebook.exit("Success!!")